In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# -----------------------------
# Utility functions
# -----------------------------
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

def cross_entropy(pred, label):
    return -np.sum(label * np.log(pred + 1e-8)) / pred.shape[0]

def accuracy(pred, label):
    return np.mean(np.argmax(pred, axis=1) == np.argmax(label, axis=1))

# -----------------------------
# Layers
# -----------------------------
class Conv2D:
    def __init__(self, num_filters, filter_size, input_depth):
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.filters = np.random.randn(num_filters, input_depth, filter_size, filter_size) * 0.1
        self.bias = np.zeros((num_filters, 1))
    
    def forward(self, x):
        self.x = x
        batch, depth, h, w = x.shape
        out_h = h - self.filter_size + 1
        out_w = w - self.filter_size + 1
        self.out = np.zeros((batch, self.num_filters, out_h, out_w))

        for b in range(batch):
            for f in range(self.num_filters):
                for i in range(out_h):
                    for j in range(out_w):
                        region = x[b, :, i:i+self.filter_size, j:j+self.filter_size]
                        self.out[b, f, i, j] = np.sum(region * self.filters[f]) + self.bias[f]
        return self.out

    def backward(self, d_out, lr):
        batch, depth, h, w = self.x.shape
        _, _, out_h, out_w = d_out.shape
        d_filters = np.zeros_like(self.filters)
        d_x = np.zeros_like(self.x)

        for b in range(batch):
            for f in range(self.num_filters):
                for i in range(out_h):
                    for j in range(out_w):
                        region = self.x[b, :, i:i+self.filter_size, j:j+self.filter_size]
                        d_filters[f] += d_out[b, f, i, j] * region
                        d_x[b, :, i:i+self.filter_size, j:j+self.filter_size] += d_out[b, f, i, j] * self.filters[f]

        self.filters -= lr * d_filters / batch
        self.bias -= lr * np.mean(d_out, axis=(0,2,3)).reshape(-1,1)
        return d_x

class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, d_out):
        return d_out * (self.x > 0)

class MaxPool2D:
    def __init__(self, size=2):
        self.size = size

    def forward(self, x):
        self.x = x
        batch, depth, h, w = x.shape
        out_h = h // self.size
        out_w = w // self.size
        self.out = np.zeros((batch, depth, out_h, out_w))
        self.argmax = np.zeros_like(x)

        for b in range(batch):
            for d in range(depth):
                for i in range(out_h):
                    for j in range(out_w):
                        region = x[b, d, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size]
                        max_val = np.max(region)
                        self.out[b, d, i, j] = max_val
                        mask = (region == max_val)
                        self.argmax[b, d, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size] = mask
        return self.out

    def backward(self, d_out):
        d_x = np.zeros_like(self.x)
        batch, depth, out_h, out_w = d_out.shape
        for b in range(batch):
            for d in range(depth):
                for i in range(out_h):
                    for j in range(out_w):
                        d_x[b, d, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size] += d_out[b, d, i, j] * self.argmax[b, d, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size]
        return d_x

class Flatten:
    def forward(self, x):
        self.input_shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, d_out):
        return d_out.reshape(self.input_shape)

class Dense:
    def __init__(self, in_size, out_size):
        self.W = np.random.randn(in_size, out_size) * 0.1
        self.b = np.zeros((1, out_size))

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, d_out, lr):
        dW = np.dot(self.x.T, d_out)
        db = np.sum(d_out, axis=0, keepdims=True)
        dx = np.dot(d_out, self.W.T)

        self.W -= lr * dW / self.x.shape[0]
        self.b -= lr * db / self.x.shape[0]
        return dx

class SoftmaxLoss:
    def forward(self, x, y):
        self.y = y
        self.probs = softmax(x)
        return cross_entropy(self.probs, y)

    def backward(self):
        return (self.probs - self.y) / self.y.shape[0]

# -----------------------------
# Build CNN
# -----------------------------
class SimpleCNN:
    def __init__(self):
        self.conv = Conv2D(8, 3, 1)
        self.relu1 = ReLU()
        self.pool = MaxPool2D(2)
        self.flatten = Flatten()
        self.fc1 = Dense(1352, 128)   # 8 * 13 * 13 = 1352
        self.relu2 = ReLU()
        self.fc2 = Dense(128, 10)
        self.loss_fn = SoftmaxLoss()

    def forward(self, x, y):
        out = self.conv.forward(x)
        out = self.relu1.forward(out)
        out = self.pool.forward(out)
        out = self.flatten.forward(out)
        out = self.fc1.forward(out)
        out = self.relu2.forward(out)
        out = self.fc2.forward(out)
        loss = self.loss_fn.forward(out, y)
        return loss, self.loss_fn.probs

    def backward(self, lr):
        d = self.loss_fn.backward()
        d = self.fc2.backward(d, lr)
        d = self.relu2.backward(d)
        d = self.fc1.backward(d, lr)
        d = self.flatten.backward(d)
        d = self.pool.backward(d)
        d = self.relu1.backward(d)
        d = self.conv.backward(d, lr)

# -----------------------------
# Train on MNIST
# -----------------------------
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train[:1000].reshape(-1, 1, 28, 28) / 255.0  # use 1000 samples for speed
y_train = to_categorical(y_train[:1000], 10)

x_test = x_test[:200].reshape(-1, 1, 28, 28) / 255.0
y_test = to_categorical(y_test[:200], 10)

cnn = SimpleCNN()
epochs = 5
lr = 0.01
batch_size = 32

for epoch in range(epochs):
    perm = np.random.permutation(len(x_train))
    x_train, y_train = x_train[perm], y_train[perm]
    loss_sum = 0

    for i in range(0, len(x_train), batch_size):
        xb = x_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]

        loss, probs = cnn.forward(xb, yb)
        cnn.backward(lr)
        loss_sum += loss

    # Evaluate
    _, preds = cnn.forward(x_test, y_test)
    acc = accuracy(preds, y_test)
    print(f"Epoch {epoch+1}, Loss={loss_sum:.4f}, Test Acc={acc:.4f}")


ModuleNotFoundError: No module named 'tensorflow'